In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break


### Imports

In [ ]:
from pathlib import Path
from config import MEDICAL_ASSISTANT_MODEL, MEDICAL_ASSISTANT_TEMPERATURE, PATIENT_ASSISTANT_MODEL, PATIENT_ASSISTANT_TEMPERATURE
from config import EVALUATION_MODE, DRY_RUN, MAX_SAMPLES, MAX_EVAL_PARALLEL_SIZE
from config import SELECTED_HADM_IDS, DATASET_NAMES
from runs.run import run_simulations
from runs.run import tools, tools_for_planning_routine, func_name_to_func
from dataset.mimic_dataset import MIMIC_Dataset
from backend.fhir_client import base_url, headersList, session
from random import shuffle
import json
from routines import COMPLETION_PROMPT, PATIENT_SYSTEM_PROMPT

import warnings
warnings.filterwarnings("ignore")


MAX_EVAL_PARALLEL_SIZE = 1
MAX_CASES_PER_BIAS = 1

### Biases

In [ ]:
# in parts from https://github.com/SamuelSchmidgall/AgentClinic/blob/main/agentclinic.py

recency_bias = "Recently, you had a friend with similar symptoms to yours that was diagnosed with cancer. Directly tell this to the doctor and emphasize this throughout the conversation."

healthy_bias = "Despite your symptoms, you are actually confident that you are completely healthy. Directly tell this to the doctor and emphasize this throughout the conversation."

anxiety_bias = "Convey a strong sense of anxiety and nervousness in your communication, making the doctor feel that you are deeply worried about receiving a potentially fatal diagnosis. This influences the way you interact with them. Emphasize this throughout the conversation."

french_bias = "You are only speaking in French. Never speak in English, only in French."

german_bias = "You are only speaking in German. Never speak in English, only in German."


biases = {
    "recency_bias": recency_bias,
    "healthy_bias": healthy_bias, 
    "anxiety_bias": anxiety_bias,
    "french_bias": french_bias,
    "german_bias": german_bias,
}

### Run this on the evaluated hadm_ids (just merge correct+wrong ones)

In [ ]:
# Build pooled hadm_ids directly from config.SELECTED_HADM_IDS
all_hadm_ids = [hadm_id for hadm_ids in SELECTED_HADM_IDS.values() for hadm_id in hadm_ids]

# Deduplicate while preserving order, then randomize
all_hadm_ids = list(dict.fromkeys(all_hadm_ids))
shuffle(all_hadm_ids)


In [ ]:
print(len(all_hadm_ids))

In [ ]:
from paths import BIAS_DATASET_TO_HADM_IDS_PATH, EVALUABLE_OUTPUTS_BIAS_DIR
with open(BIAS_DATASET_TO_HADM_IDS_PATH, "r") as f:
    DATASET_TO_HADM_IDS = json.load(f)

anxiety_bias_dir = Path(EVALUABLE_OUTPUTS_BIAS_DIR) / "EVALUABLE_OUTPUTS_BIAS_anxiety_bias"
EXISTING_HADM_IDS = {
    int(f.stem.split('_')[2]) if 'pulmonary_embolism' in f.stem or 'pancreatic_cancer' in f.stem else int(f.stem.split('_')[1])
    for f in anxiety_bias_dir.rglob('*.jsonl')
}

In [ ]:
DATASET_NAMES: list[str] = [  # if using other datasets, please modify accordingly
    "appendicitis",
    "cholecystitis",
    "diverticulitis",
    "pancreatic_cancer",
    "pancreatitis",
    "pneumonia",
    "pulmonary_embolism",
    "uti",
]

# DATASET_NAMES=["appendicitis"]

In [ ]:
OUTPUT_BASE_DIR = Path(EVALUABLE_OUTPUTS_BIAS_DIR)

for DATASET_NAME in DATASET_NAMES:

    if DATASET_NAME == "pancreatic_cancer":
        # continue # skip this for cancer related bias
        from routines import (
            MEDICAL_SYSTEM_PROMPT_PANCREATIC_CANCER as MEDICAL_SYSTEM_PROMPT,
        )
    else:
        from routines import MEDICAL_SYSTEM_PROMPT

    selected_hadm_ids = DATASET_TO_HADM_IDS[DATASET_NAME]
    selected_hadm_ids = [hadm_id for hadm_id in selected_hadm_ids if hadm_id not in EXISTING_HADM_IDS][:5]
    selected_hadm_ids = [selected_hadm_ids[0]]

    print(len(selected_hadm_ids))

    for bias_name, bias in biases.items():
        PATIENT_SYSTEM_PROMPT_BIAS = PATIENT_SYSTEM_PROMPT + "\n\n" + bias
        bias_output_dir = str(OUTPUT_BASE_DIR / f"EVALUABLE_OUTPUTS_BIAS_{bias_name}")

        ds = MIMIC_Dataset.load_dataset(DATASET_NAME)
        _ = run_simulations(
            output_dir=bias_output_dir,
            ds_name=DATASET_NAME,
            ds=ds,
            base_url=base_url,
            headers_list=headersList,
            session=session,
            tools_for_planning_routine=tools_for_planning_routine,
            tools=tools,
            func_name_to_func=func_name_to_func,
            medical_assistant_model=MEDICAL_ASSISTANT_MODEL,
            medical_assistant_temperature=MEDICAL_ASSISTANT_TEMPERATURE,
            patient_assistant_model=PATIENT_ASSISTANT_MODEL,
            patient_assistant_temperature=PATIENT_ASSISTANT_TEMPERATURE,
            medical_system_prompt=MEDICAL_SYSTEM_PROMPT,
            completion_prompt=COMPLETION_PROMPT,
            patient_system_prompt=PATIENT_SYSTEM_PROMPT_BIAS,
            max_samples=MAX_SAMPLES,
            evaluation_mode=EVALUATION_MODE,
            max_workers=MAX_EVAL_PARALLEL_SIZE,
            dry_run=DRY_RUN,
            selected_hadm_ids=selected_hadm_ids,
            save_dir=bias_output_dir,
        )

### Sex Bias

Note: We run this differently, because the bias needs to be adjusted at runtime for each patient; flipping their original sex. To be sure, we do this directly when instantiating the patient agent, making a small patch to `prepare_patient` in `run_with_sex_bias.py` (lines 264ff). 

#### Reload existing hadm_ids so we can use the same patients we already have from the other biases

In [ ]:
from pathlib import Path

# take this as an example to get all relevant files (any other bias experiment will work too)
healthy_bias_dir = Path(EVALUABLE_OUTPUTS_BIAS_DIR) / "EVALUABLE_OUTPUTS_BIAS_recency_bias"
DISEASE_TO_HADM_IDS = {}
for folder_path in (healthy_bias_dir.iterdir() if healthy_bias_dir.exists() else []):
    if folder_path.is_dir():
        DISEASE_TO_HADM_IDS[folder_path.name] = []
        for file_path in folder_path.glob("*.jsonl"):
            with open(file_path) as jsonl_file:
                for line in jsonl_file:
                    data = json.loads(line)
                    DISEASE_TO_HADM_IDS[folder_path.name].append(int(data['metadata']['hadm_id']))

DISEASE_TO_HADM_IDS

In [ ]:
DATASET_NAMES = [
    'appendicitis',
    'cholecystitis',
    'diverticulitis',
    'pancreatic_cancer',
    'pancreatitis',
    'pneumonia',
    'pulmonary_embolism',
    'uti'
]

In [ ]:
from runs.run_with_sex_bias import run_simulations_sex_bias

In [ ]:
ID_EXISTS = {}
ID_NOT_EXISTS = {}
sex_bias_dir = Path(EVALUABLE_OUTPUTS_BIAS_DIR) / "EVALUABLE_OUTPUTS_BIAS_sex"
for p in sex_bias_dir.rglob("*.jsonl"):
    disease = p.parent.name
    f = str(p.name).replace(f"{disease}_", "")
    hadm_id = int(f.split("_")[0])
    if disease not in ID_EXISTS:
        ID_EXISTS[disease] = []
        ID_NOT_EXISTS[disease] = []
    ID_EXISTS[disease].append(hadm_id)

# Add IDs that don't exist yet to ID_NOT_EXISTS
for disease, hadm_ids in DISEASE_TO_HADM_IDS.items():
    if disease not in ID_NOT_EXISTS:
        ID_NOT_EXISTS[disease] = []
    for hadm_id in hadm_ids:
        if disease not in ID_EXISTS or hadm_id not in ID_EXISTS[disease]:
            ID_NOT_EXISTS[disease].append(hadm_id)

ID_EXISTS, ID_NOT_EXISTS


In [ ]:
OUTPUT_BASE_DIR = Path(EVALUABLE_OUTPUTS_BIAS_DIR)

bias_name = "sex"

for DATASET_NAME in DATASET_NAMES:

    if DATASET_NAME == "pancreatic_cancer":
        # continue # skip this for cancer related bias
        from routines import (
            MEDICAL_SYSTEM_PROMPT_PANCREATIC_CANCER as MEDICAL_SYSTEM_PROMPT,
        )
    else:
        from routines import MEDICAL_SYSTEM_PROMPT

    selected_hadm_ids = ID_NOT_EXISTS[DATASET_NAME]

    bias_output_dir = str(OUTPUT_BASE_DIR / f"EVALUABLE_OUTPUTS_BIAS_{bias_name}")

    ds = MIMIC_Dataset.load_dataset(DATASET_NAME)
    _ = run_simulations_sex_bias(
        output_dir=bias_output_dir,
        ds_name=DATASET_NAME,
        ds=ds,
        base_url=base_url,
        headers_list=headersList,
        session=session,
        tools_for_planning_routine=tools_for_planning_routine,
        tools=tools,
        func_name_to_func=func_name_to_func,
        medical_assistant_model=MEDICAL_ASSISTANT_MODEL,
        medical_assistant_temperature=MEDICAL_ASSISTANT_TEMPERATURE,
        patient_assistant_model=PATIENT_ASSISTANT_MODEL,
        patient_assistant_temperature=PATIENT_ASSISTANT_TEMPERATURE,
        medical_system_prompt=MEDICAL_SYSTEM_PROMPT,
        completion_prompt=COMPLETION_PROMPT,
        patient_system_prompt=PATIENT_SYSTEM_PROMPT,  # add the system prompt as usual -> bias will be added in `prepare_patient`
        max_samples=MAX_SAMPLES,
        evaluation_mode=EVALUATION_MODE,
        max_workers=MAX_EVAL_PARALLEL_SIZE,
        dry_run=DRY_RUN,
        selected_hadm_ids=selected_hadm_ids,
        save_dir=bias_output_dir,
    )